# Real Student Stress Prediction — Notebook Demonstration

This notebook loads the **two final fitted pipelines** saved by Notebook 07. It makes new predictions for existing students in the reserved test set without retraining. It includes **correct and incorrect** cases when available and displays the actual reported stress level for comparison.

**Note:** Predictions are exploratory and should not be used as clinical or psychological diagnoses. For real-world prediction, never show a ground-truth value unless independently known.

In [1]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
DATA = ROOT / 'data'
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
MODELS = ROOT / 'models'
for p in [TABLES, FIGURES, MODELS, DATA/'processed']:
    p.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})


In [2]:
import joblib
meta=json.loads((DATA/'processed'/'experiment_config.json').read_text())
specs=json.loads((MODELS/'demo_model_selection.json').read_text())
df=pd.read_csv(DATA/'processed'/'dataset2_cleaned.csv')
pred_records=pd.read_csv(TABLES/'heldout_predictions.csv')
models={task:joblib.load(MODELS/f'best_model_{task}.pkl') for task in ['3class','5class']}
print('Saved models loaded:')
for task in models:
    print(task,':',specs[task]['model'],'with',specs[task]['n_features'],'features')


Saved models loaded:
3class : Decision Tree with 7 features
5class : Decision Tree with 15 features


## 1. Select an actual held-out student

Select either task and a row index from the reserved test cases. The notebook will print their actual survey response and the model prediction. To try a different case, change `student_row` to another row index printed below.

In [3]:
TASK='3class'  # Change to '5class' for the five-level model.
available=pred_records[pred_records['task'].eq(TASK)]
print('First 10 real test row indices:',available['row_index'].head(10).tolist())
student_row=int(available['row_index'].iloc[0])
spec=specs[TASK]
columns=meta['seven_features'] if spec['n_features']==7 else meta['fifteen_features']
student=df.loc[[student_row],columns]
actual=int(df.loc[student_row,meta['target']])
actual={1:'Low',2:'Low',3:'Medium',4:'High',5:'High'}[actual] if TASK=='3class' else actual
prediction=models[TASK].predict(student)[0]
print('Student row index:',student_row)
display(student.T.rename(columns={student_row:'Student response'}))
print('Reported stress:',actual,'| Predicted stress:',prediction)
print('Correct?',bool(actual==prediction))
assert str(prediction)==str(available.loc[available.row_index.eq(student_row),'predicted'].iloc[0])


First 10 real test row indices: [0, 5, 8, 15, 27, 32, 37, 38, 40, 54]
Student row index: 0


,Student response
Average_sleep_duration_per_night,5-6 hrs
Sleep_quality,Good
Hours_spent_daily_on_academic_activities,>4 hrs
Physical_activity_per_week,Moderate
Use_of_electronic_devices_before_bed,Yes
Night_time_caffeine_consumption,Yes
Diet_quality,Average


Reported stress: High | Predicted stress: Low
Correct? False


## 2. Show a real correct and an incorrect prediction (when available)

This is **not a cherry-picked accuracy evaluation**; final metrics are computed for all held-out cases in Notebook 07. These examples simply demonstrate model behavior.

In [4]:
for task in ['3class','5class']:
    part=pred_records[pred_records.task.eq(task)]
    print('\n',task,'— selected',specs[task]['model'])
    display(pd.concat([part[part.correct.eq(True)].head(1),part[part.correct.eq(False)].head(1)]))



 3class — selected Decision Tree


,task,row_index,actual,predicted,correct
3,3class,15,Low,Low,True
0,3class,0,High,Low,False



 5class — selected Decision Tree


,task,row_index,actual,predicted,correct
61,5class,5,1,1,True
60,5class,0,5,2,False


## 3. Predict your own stress category — dropdown form

Choose your answers using the dropdown boxes below, then press **Predict Stress**. No code editing or retraining is required.

- **3 classes (recommended demo):** 7 lifestyle questions → Low, Medium, or High.
- **5 classes (optional):** 15 questions → stress level 1–5.
- The choices match values in **Dataset 2**; numerical values remain numerical, not strings.
- Your answers are used only for the current prediction; the notebook does not save them or retrain the model.

**Important:** This is a classroom demonstration, not a health assessment. The 3-class model's held-out accuracy was only 41.67%, below the 50% majority-class baseline.


In [ ]:
# Selectable student survey form for VS Code/Jupyter notebooks.
# Requires: python -m pip install ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Friendly labels; actual column names and allowed answers come from Dataset 2.
questions = {
    "Average_sleep_duration_per_night": "Average sleep duration per night",
    "Sleep_quality": "Sleep quality",
    "Hours_spent_daily_on_academic_activities": "Daily hours of academic activities",
    "Physical_activity_per_week": "Weekly physical activity",
    "Use_of_electronic_devices_before_bed": "Use electronic devices before bed?",
    "Night_time_caffeine_consumption": "Drink caffeine at night?",
    "Diet_quality": "Diet quality",
    "Overwhelmed_by_academic_workload": "Feeling overwhelmed by academic workload",
    "Financial_stress_level": "Financial stress level (1–5)",
    "Strength_of_social_support": "Social support",
    "Bedtime_on_regular_nights": "Usual bedtime",
    "Daytime_sleepiness_during_classes": "Daytime sleepiness in class",
    "Focus_on_academic_tasks": "Focus on academic tasks",
    "Type_of_residence": "Living arrangement",
    "Current_Study_Year": "Current study year",
}

# Preferred display order, but NEVER allow choices absent from the actual dataset.
preferred_order = {
    "Average_sleep_duration_per_night": ["<5 hrs", "5-6 hrs", "7-8 hrs", ">8 hrs"],
    "Sleep_quality": ["Poor", "Average", "Good"],
    "Hours_spent_daily_on_academic_activities": ["<1 hrs", "1-2 hrs", "2-3 hrs", "3-4 hrs", ">4 hrs"],
    "Physical_activity_per_week": ["Low", "Moderate", "High"],
    "Use_of_electronic_devices_before_bed": ["No", "Yes"],
    "Night_time_caffeine_consumption": ["No", "Yes"],
    "Diet_quality": ["Poor", "Average", "Good"],
    "Overwhelmed_by_academic_workload": ["Never", "Rarely", "Sometimes", "Often", "Always"],
    "Financial_stress_level": [1, 2, 3, 4, 5],
    "Strength_of_social_support": ["Low", "Moderate", "High"],
    "Bedtime_on_regular_nights": ["Before 10 PM", "10-11 PM", "11-12 AM", "After 12 AM"],
    "Daytime_sleepiness_during_classes": ["Never", "Rarely", "Often", "Always"],
    "Focus_on_academic_tasks": ["Never", "Rarely", "Sometimes", "Often", "Always"],
    "Type_of_residence": ["On-Campus", "Off-Campus", "With Family"],
    "Current_Study_Year": [1, 2, 3, 4],
}

def choices_from_data(column):
    observed = list(df[column].dropna().unique())
    # Cast numpy scalars to Python ints for compatibility with ipywidgets / DataFrame.
    observed = [x.item() if isinstance(x, np.generic) else x for x in observed]
    preferred = preferred_order[column]
    return [v for v in preferred if v in observed] + [v for v in observed if v not in preferred]

controls = {}
for column in meta["fifteen_features"]:
    values = choices_from_data(column)
    # Initial values are defaults only; change each answer before clicking Predict.
    initial = df[column].mode(dropna=True).iloc[0]
    if isinstance(initial, np.generic):
        initial = initial.item()
    controls[column] = widgets.Dropdown(
        options=[(str(value), value) for value in values],
        value=initial,
        layout=widgets.Layout(width="100%"),
        style={"description_width": "initial"},
    )

model_choice = widgets.Dropdown(
    options=[("3 classes — 7 lifestyle questions", "3class"),
             ("5 classes — 15 questions", "5class")],
    value="3class",
    description="Prediction task:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="490px"),
)

fields_panel = widgets.GridBox(
    children=[],
    layout=widgets.Layout(
        grid_template_columns="repeat(2, minmax(275px, 1fr))",
        grid_gap="12px 20px",
        width="100%",
    ),
)

predict_button = widgets.Button(description="Predict Stress", button_style="success", icon="check")
reset_button = widgets.Button(description="Reset answers", icon="refresh")
result_panel = widgets.Output()
notice = widgets.HTML(
    value="<small><b>Learning demo only:</b> Predictions have limited reliability and are not a diagnosis.</small>"
)

def active_columns():
    task = model_choice.value
    return meta["seven_features"] if specs[task]["n_features"] == 7 else meta["fifteen_features"]

def update_form(change=None):
    cards = []
    for col in active_columns():
        cards.append(widgets.VBox([
            widgets.HTML(value=f"<b>{questions[col]}</b>"),
            controls[col],
        ], layout=widgets.Layout(width="100%")))
    fields_panel.children = tuple(cards)
    result_panel.clear_output()

def reset_form(button=None):
    for col in active_columns():
        initial = df[col].mode(dropna=True).iloc[0]
        if isinstance(initial, np.generic):
            initial = initial.item()
        controls[col].value = initial
    result_panel.clear_output()

def predict_stress(button):
    task = model_choice.value
    cols = active_columns()
    # Values remain EXACTLY the same categories/types used in the training dataset.
    input_frame = pd.DataFrame([{col: controls[col].value for col in cols}], columns=cols)
    result = models[task].predict(input_frame)[0]
    with result_panel:
        clear_output(wait=True)
        if task == "3class":
            print(f"Predicted stress category: {result}")
        else:
            print(f"Predicted stress level: {int(result)} (out of 5)")
        print(f"Model: Decision Tree | Inputs: {len(cols)}")
        print("Note: This estimate is for a classroom demonstration, not a medical assessment.")

model_choice.observe(update_form, names="value")
predict_button.on_click(predict_stress)
reset_button.on_click(reset_form)
update_form()

display(widgets.VBox([
    widgets.HTML("<h3>Student Stress Prediction — Interactive Demo</h3>"),
    model_choice,
    widgets.HTML("<p>Select your answers using the dropdowns:</p>"),
    fields_panel,
    widgets.HBox([predict_button, reset_button]),
    result_panel,
    notice,
], layout=widgets.Layout(width="100%")))


### Running the dropdown form in VS Code

1. Open `demo.ipynb`, choose your `.venv` kernel, and run all cells.
2. Scroll to **Student Stress Prediction — Interactive Demo**.
3. Choose **3 classes** (7 questions) or **5 classes** (15 questions).
4. Use each dropdown to enter your own answers and click **Predict Stress**.
5. To reuse the form, change any answer and click the button again; **Reset answers** restores the dataset's most common response for each question.

If you see `ModuleNotFoundError: No module named 'ipywidgets'`, run `%pip install ipywidgets` in a notebook cell, then restart the kernel and rerun the notebook. Make sure the **Python** and **Jupyter** VS Code extensions are installed. This works in VS Code's notebook view; you do not need to launch JupyterLab.
